# Welcome to Colab!

## DeepGait Project Summary from DeepGait_POC.pdf

### Project Objective:
To demonstrate a functional, ready-to-deploy system for Gait Recognition using unique walking patterns for identification, establishing system feasibility while avoiding over-engineering.

### System Architecture:
#### Phase 1: Skeletal Extraction (The "Eye")
*   **Tool:** Utilizes BlazePose to identify 17 key joint points from standard video or CCTV footage in real-time.
*   **Normalization Pipeline:** Raw data undergoes:
    *   **Centering:** Aligning the body center to a standardized origin.
    *   **Scaling:** Standardizing all bodies to a uniform size.
    *   **Cleaning:** Repairing missing joints based on confidence scores.

#### Phase 2: Identification (The "Brain")
*   **Core Component:** Spatio-Temporal Graph Convolutional Network (ST-GCN).
    *   **Spatial Logic:** Recognizes natural joint connections as a graph.
    *   **Temporal Logic:** Tracks joint movement across a 64-frame window to capture motion dynamics.
*   **Infrastructure:** Built upon the OpenGait framework and deployed on a dedicated server.

### Input Data Format and Specific Requirements:
*   **Joint Points:** Optimized to 17 key joints for simplified processing.
*   **Dimensionality:** The biometric signature is compressed to 128 dimensions.
*   **Coordinates:** Prioritizes 2D (X, Y) coordinates to reduce computational complexity while maintaining recognition accuracy.
*   **Temporal Window:** Movement tracked across a 64-frame window.

### Readiness for Further Development:
*   The POC focuses on achieving a functional delivery within a timeframe, indicating a streamlined approach with specific scope reductions.
*   The ST-GCN model is explicitly stated as the core for identification, with clear definitions of its spatial and temporal logic, suggesting a foundation is in place.
*   The system leverages BlazePose for skeletal extraction and OpenGait for its framework, implying a degree of component integration and readiness for module development related to the ST-GCN.

# חיבור גוגל דרייב למחברת


# פתיחת הקבצים והורדה לתיקייה מקומית

מקומית: הרץ `pip install -e .` משורש הפרויקט (סביבת `.venv`) כדי ש־`import deep_gait` יעבוד. הארכיון נשמר ב־`data/raw/` והחילוץ ל־`data/processed/casia_b_hrnet/`.

In [ ]:
from deep_gait.dataset import mount_google_drive_if_colab

mount_google_drive_if_colab()

In [ ]:
from deep_gait.dataset import download_casia_if_local

# Optional: pass a Drive URL or file ID, or set GOOGLE_DRIVE_FILE_URL_OR_ID
download_casia_if_local()

In [ ]:
from deep_gait.dataset import extract_casia_archive

extract_casia_archive()

## יישום בלוק קונבולוציה זמנית

### Subtask:
פיתוח מודול PyTorch עבור רכיב הקונבולוציה הזמנית כמתואר בארכיטקטורת ST-GCN, הכולל קונבולוציות חד-ממדיות לאורך מימד הזמן.


**Reasoning**:
I will now implement the `TemporalConv` module as described in the instructions, using PyTorch's `nn.Conv1d` for the temporal convolution, followed by batch normalization and a ReLU activation function. This module is a core component of the ST-GCN architecture, focusing on processing features along the time dimension.



In [ ]:
import torch
import torch.nn as nn

class TemporalConv(nn.Module):
    """
    Temporal Convolutional module for ST-GCN.
    Applies a 1D convolution along the time dimension.
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(TemporalConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding
            ),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # x should be of shape (batch_size, in_channels, time_steps)
        return self.conv(x)

print("TemporalConv module defined successfully.")

## יישום בלוק קונבולוציה גרפית מרחבית

### Subtask:
פיתוח מודול PyTorch עבור רכיב הקונבולוציה הגרפית המרחבית, שיטפל בקונבולוציות מבוססות גרף על המימד המרחבי. זה חיוני עבור ST-GCN.


**Reasoning**:
I need to implement the `SpatialGraphConv` module. This involves defining a PyTorch module with a 2D convolution (`nn.Conv2d`) for feature transformation, a learnable adjacency matrix (`nn.Parameter`), batch normalization, and a ReLU activation function. The key challenge is to correctly apply the graph multiplication with the adjacency matrix on the spatial dimension of the input features, which typically involves reshaping and matrix multiplication before applying the BatchNorm and ReLU.



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TemporalConv(nn.Module):
    """
    Temporal Convolutional module for ST-GCN.
    Applies a 1D convolution along the time dimension.
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(TemporalConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding
            ),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # x should be of shape (batch_size, in_channels, time_steps)
        return self.conv(x)


class SpatialGraphConv(nn.Module):
    """
    Spatial Graph Convolutional module for ST-GCN.
    """
    def __init__(self, in_channels, out_channels, num_nodes):
        super(SpatialGraphConv, self).__init__()
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=(1, 1),
            stride=(1, 1),
            padding=(0, 0)
        )
        self.A = nn.Parameter(torch.ones(num_nodes, num_nodes), requires_grad=True)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # x shape: (N, C, T, V)
        x_conv = self.conv(x)

        # einsum avoids the permute/matmul dimension mismatch
        # Multiplies V dimension of x_conv with the VxV Adjacency matrix
        output = torch.einsum('nctv,vw->nctw', x_conv, self.A)

        output = self.bn(output)
        output = self.relu(output)
        return output


class STGCN_Block(nn.Module):
    def __init__(self, in_channels, out_channels, num_nodes, temporal_kernel=9):
        super().__init__()
        self.spatial = SpatialGraphConv(in_channels, out_channels, num_nodes)

        self.temporal = TemporalConv(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=temporal_kernel,
            padding=temporal_kernel // 2
        )

    def forward(self, x):
        # 1. Spatial Pass: (N, C, T, V)
        x = self.spatial(x)

        # 2. Reshape for TemporalConv (Conv1d expects 3D tensor: Batch, Channels, Time)
        N, C, T, V = x.shape
        x = x.permute(0, 3, 1, 2).reshape(N * V, C, T)

        # 3. Temporal Pass
        x = self.temporal(x)

        # 4. Reshape back to 4D tensor: (N, C, T, V)
        x = x.reshape(N, V, C, T).permute(0, 2, 3, 1).contiguous()
        return x


class DeepGait_STGCN(nn.Module):
    def __init__(self, num_nodes=17, in_channels=2, num_classes=128):
        super().__init__()
        # Stacking ST-GCN blocks
        self.block1 = STGCN_Block(in_channels=in_channels, out_channels=64, num_nodes=num_nodes)
        self.block2 = STGCN_Block(in_channels=64, out_channels=64, num_nodes=num_nodes)

        # Classifier / Compression layer
        self.fc = nn.Linear(64 * num_nodes, num_classes)

    def forward(self, x):
        # x shape expected: (Batch, Channels, Time, Vertices)
        x = self.block1(x)
        x = self.block2(x)

        # Global Average Pooling over the temporal dimension
        x = F.avg_pool2d(x, (x.size(2), 1))

        # Flatten and output
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


def validate_forward_pass():
    # Setup model based on POC constraints
    model = DeepGait_STGCN(num_nodes=17, in_channels=2, num_classes=128)

    # N=16 (Batch), C=2 (2D coords), T=64 (Frames), V=17 (Joints)
    dummy_input = torch.randn(16, 2, 64, 17)

    print("Running Forward Pass Validation...")
    print(f"Input shape:  {dummy_input.shape}")

    output = model(dummy_input)

    print(f"Output shape: {output.shape} -> Expected: (16, 128)")
    if output.shape == (16, 128):
        print("Validation Successful! Shapes match POC requirements.")
    else:
        print("Validation Failed. Check dimensions.")

if __name__ == "__main__":
    validate_forward_pass()

TEST

In [ ]:
def validate_forward_pass():
    model = DeepGait_STGCN()

    # N=16 (Batch), C=2 (2D coords), T=64 (Frames), V=17 (Joints)
    dummy_input = torch.randn(16, 2, 64, 17)

    print(f"Input shape:  {dummy_input.shape}")
    output = model(dummy_input)
    print(f"Output shape: {output.shape} -> Expected: (16, 128)")

validate_forward_pass()

## הרכבת מודל ST-GCN מלא

### Subtask:
שילוב בלוקי הקונבולוציה הזמניים והמרחביים שיושמו, יחד עם שכבות נחוצות אחרות (כגון נורמליזציה, פונקציות הפעלה, שכבות pooling/readout), כדי לבנות את מודל ה-ST-GCN המלא כמתואר במסמך ה-POC של DeepGait.


**Reasoning**:
I need to integrate the previously defined `TemporalConv` and `SpatialGraphConv` modules into a full `ST-GCN` model. This involves creating an `STGCN_Block` class that combines these two convolutional operations and then assembling multiple `STGCN_Block` instances with pooling and a final fully connected layer into the main `ST_GCN` model, as specified by the subtask instructions.



In [ ]:
import torch
import torch.nn as nn

# Assuming TemporalConv and SpatialGraphConv are already defined and available
# from previous steps in the kernel context. If not, they would need to be
# included here for the code to run independently.

class STGCN_Block(nn.Module):
    """
    Combines SpatialGraphConv and TemporalConv modules.
    """
    def __init__(self, in_channels, out_channels, num_nodes, temporal_kernel_size=5):
        super(STGCN_Block, self).__init__()
        self.num_nodes = num_nodes
        self.out_channels = out_channels
        temporal_padding = temporal_kernel_size // 2

        self.spatial_conv = SpatialGraphConv(in_channels, out_channels, num_nodes)
        self.temporal_conv = TemporalConv(out_channels, out_channels, temporal_kernel_size, padding=temporal_padding)

    def forward(self, x):
        # x shape: (batch_size, in_channels, time_steps, num_nodes)
        batch_size, in_channels, time_steps, num_nodes = x.shape

        # 1. Apply Spatial Graph Convolution
        x_spatial = self.spatial_conv(x)
        # x_spatial shape: (batch_size, out_channels, time_steps, num_nodes)

        # 2. Reshape for Temporal Convolution
        # Permute to (batch_size, num_nodes, out_channels, time_steps)
        # Then view as (batch_size * num_nodes, out_channels, time_steps)
        x_temporal_input = x_spatial.permute(0, 3, 1, 2).contiguous().view(
            batch_size * num_nodes, self.out_channels, time_steps
        )

        # 3. Apply Temporal Convolution
        x_temporal_output = self.temporal_conv(x_temporal_input)
        # x_temporal_output shape: (batch_size * num_nodes, out_channels, time_steps)

        # 4. Reshape back to (batch_size, out_channels, time_steps, num_nodes)
        x_out = x_temporal_output.view(
            batch_size, num_nodes, self.out_channels, time_steps
        ).permute(0, 2, 3, 1).contiguous()

        return x_out


class ST_GCN(nn.Module):
    """
    Full ST-GCN model as described in the DeepGait POC.pdf.
    """
    def __init__(self, in_channels, num_nodes, num_classes):
        super(ST_GCN, self).__init__()

        # Define a sequence of STGCN Blocks
        self.stgcn_blocks = nn.Sequential(
            STGCN_Block(in_channels, 64, num_nodes, temporal_kernel_size=5),
            STGCN_Block(64, 128, num_nodes, temporal_kernel_size=5)
            # Further blocks could be added here based on specific requirements
        )

        # Global pooling layer to aggregate features across time and nodes
        # The output of the last STGCN block is (B, C_out, T, V), where C_out is 128.
        # AdaptiveAvgPool2d((1,1)) reduces T and V dimensions to 1, resulting in (B, 128, 1, 1).
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        # Final fully connected layer to map pooled features to num_classes
        # After flattening, the pooled features are 128-dimensional.
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        # x shape: (batch_size, in_channels, time_steps, num_nodes)

        # Pass through STGCN blocks
        x = self.stgcn_blocks(x)
        # Output shape from the last STGCN_Block: (batch_size, 128, time_steps, num_nodes)

        # Apply global pooling
        x = self.pool(x)
        # Output shape: (batch_size, 128, 1, 1)

        # Flatten the pooled features
        x = x.view(x.size(0), -1)
        # Output shape: (batch_size, 128)

        # Pass through the final fully connected layer
        x = self.fc(x)
        # Output shape: (batch_size, num_classes)

        return x

print("STGCN_Block and ST_GCN modules defined successfully.")

## אימות מעבר קדימה (Forward Pass)

### Subtask:
יצירת נתוני קלט דמה בעלי הצורה הצפויה והעברתם דרך מודל ה-ST-GCN המורכב. ודא שהמודל מייצר פלט בעל הצורה הנכונה ושאין שגיאות במהלך חישוב המעבר קדימה.


**Reasoning**:
I need to create dummy input data, instantiate the ST-GCN model, perform a forward pass, and print the output shape to verify the model's functionality and output dimensions as per the subtask instructions.



In [ ]:
import torch
import torch.nn as nn

class SpatialGraphConv(nn.Module):
    """
    Spatial Graph Convolutional module for ST-GCN.
    Applies a 1x1 convolution for feature transformation and then multiplies
    by a learnable adjacency matrix.
    """
    def __init__(self, in_channels, out_channels, num_nodes):
        super(SpatialGraphConv, self).__init__()
        # 1. Define a 2D convolution layer (1x1 kernel) for feature transformation
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=(1, 1), # (time_kernel, spatial_kernel)
            stride=(1, 1),
            padding=(0, 0)
        )
        # 2. Initialize a nn.Parameter for the learnable adjacency matrix
        # Initializing with ones can represent an initial assumption of full connectivity,
        # which the model can then learn to prune or strengthen.
        # In ST-GCN, multiple adjacency matrices are often used, here we define a single one.
        self.A = nn.Parameter(torch.ones(num_nodes, num_nodes), requires_grad=True)

        # 3. Include BatchNorm2d and ReLU activation
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # x is expected to be of shape (batch_size, in_channels, time_steps, num_nodes)

        # 1. Apply the 1x1 convolution for feature transformation
        # This transforms the input channels from in_channels to out_channels
        # Output shape: (batch_size, out_channels, time_steps, num_nodes)
        x_conv = self.conv(x)

        # 2. Implement the graph convolution logic by multiplying with the adjacency matrix
        # x_conv shape: (N, C_out, T, V)
        # self.A shape: (V, V)
        # For torch.matmul(input, other), the last dimension of input must match the second-to-last dimension of other.
        # Here, V matches V, so the multiplication is valid.
        # The result will have shape (N, C_out, T, V).
        output = torch.matmul(x_conv, self.A)

        # 3. Apply BatchNorm2d and ReLU activation
        output = self.bn(output)
        output = self.relu(output)

        return output

print("SpatialGraphConv module defined successfully.")

**Reasoning**:
Since the `SpatialGraphConv` module has been corrected and redefined, I will now re-run the forward pass verification step to ensure the entire `ST_GCN` model works correctly with the fixed spatial convolution and produces the expected output shape.



In [ ]:
import torch

# 1. Define example values based on DeepGait POC document
in_channels = 2  # X,Y coordinates
num_nodes = 17   # 17 key joints
num_classes = 10 # Example: 10 different gait patterns/identities for classification
batch_size = 16  # Example batch size
time_steps = 64  # 64-frame window

# 2. Create a dummy input tensor
# Shape: (batch_size, in_channels, time_steps, num_nodes)
dummy_input = torch.randn(batch_size, in_channels, time_steps, num_nodes)

print(f"Dummy input shape: {dummy_input.shape}")

# 3. Instantiate the ST_GCN model
# Assuming ST_GCN class is defined in a previous cell and accessible
model = ST_GCN(in_channels=in_channels, num_nodes=num_nodes, num_classes=num_classes)

print("ST_GCN model instantiated successfully.")

# 4. Pass the dummy_input through the ST_GCN model
output = model(dummy_input)

# 5. Print the shape of the output tensor
print(f"Output shape from ST_GCN model: {output.shape}")

# Expected output shape: (batch_size, num_classes)
expected_shape = (batch_size, num_classes)
if output.shape == expected_shape:
    print(f"Output shape matches expected shape: {expected_shape}")
else:
    print(f"WARNING: Output shape {output.shape} does not match expected shape {expected_shape}")

## שילוב עם צינור אימון בסיסי

### Subtask:
הגדרת לולאת אימון מינימלית. זה כולל הגדרת פונקציית הפסד (לדוגמה, CrossEntropyLoss לסיווג או MSELoss לרגולציה), אופטימיזטור (לדוגמה, Adam) והכנת תוויות דמה כדי לדמות תהליך אימון.


**Reasoning**:
I will implement a minimal training loop by defining a loss function (CrossEntropyLoss) and an optimizer (Adam), generating dummy target labels, and then performing a single forward pass, loss calculation, backward pass, and optimizer step to demonstrate the training process.



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Ensure the ST_GCN model is accessible from previous cells
# (assuming it's defined and `model` is an instance of ST_GCN)

# 1. Define a loss function for classification
criterion = nn.CrossEntropyLoss()
print("Loss function (CrossEntropyLoss) defined.")

# 2. Define an optimizer and pass the model's parameters to it
# Using Adam optimizer with a learning rate
optimizer = optim.Adam(model.parameters(), lr=0.001)
print("Optimizer (Adam) defined.")

# 3. Generate dummy target labels
# Labels should be integers representing class indices (0 to num_classes - 1)
dummy_labels = torch.randint(0, num_classes, (batch_size,))
print(f"Dummy labels shape: {dummy_labels.shape}")

# 4. Simulate a single training step
print("\nSimulating a single training step...")

# a. Set the optimizer's gradients to zero
optimizer.zero_grad()

# b. Perform a forward pass
output = model(dummy_input)

# c. Calculate the loss
loss = criterion(output, dummy_labels)

# d. Perform a backward pass
loss.backward()

# e. Update the model's parameters
optimizer.step()

# 5. Print the calculated loss
print(f"Loss after one training step: {loss.item()}")
print("Single training step completed successfully.")

## Final Task

### Subtask:
Summarize the completed ST-GCN model implementation and its readiness for further development based on the DeepGait_POC.pdf.


## Summary:

### Data Analysis Key Findings

*   **DeepGait Project and ST-GCN Architecture**: The `DeepGait_POC.pdf` was successfully analyzed, outlining the project's objective to demonstrate a functional gait recognition system. The system architecture involves BlazePose for skeletal extraction (17 key joints, normalized X, Y coordinates, 64-frame window) and a Spatio-Temporal Graph Convolutional Network (ST-GCN) for identification. The ST-GCN uses spatial logic for joint connections and temporal logic for motion dynamics.
*   **Temporal Convolution Module (`TemporalConv`)**: A PyTorch module for temporal convolution was successfully implemented. It consists of a `Conv1d` layer for feature transformation along the time dimension, followed by `BatchNorm1d` and `ReLU` activation.
*   **Spatial Graph Convolution Module (`SpatialGraphConv`)**: A PyTorch module for spatial graph convolution was successfully implemented. It applies a `Conv2d` (1x1 kernel) for feature transformation, multiplies with a learnable adjacency matrix, and includes `BatchNorm2d` and `ReLU` activation.
    *   *Correction Note*: An initial `RuntimeError` occurred due to a dimension mismatch in the `torch.matmul` operation within the `SpatialGraphConv`'s `forward` pass, where `x_graph` (`17408x64`) and `self.A` (`17x17`) could not be multiplied. This was corrected by adjusting the tensor permutation to ensure the last dimension of `x_conv` (representing `num_nodes`) aligned correctly for matrix multiplication with the adjacency matrix.
*   **Full ST-GCN Model Assembly (`STGCN_Block` and `ST_GCN`)**: The `STGCN_Block` module, combining the `SpatialGraphConv` and `TemporalConv` layers, was successfully defined. The complete `ST_GCN` model was then assembled using a sequence of `STGCN_Block`s, followed by `AdaptiveAvgPool2d` for global pooling and a `Linear` layer for classification, matching the specifications.
*   **Forward Pass Validation**: After the necessary correction in `SpatialGraphConv`, the `ST_GCN` model successfully processed dummy input data (shape: `16` batches, `2` channels, `64` time steps, `17` nodes). The forward pass completed without errors, yielding an output tensor of shape `(16, 10)`, which aligns with the expected `(batch_size, num_classes)`.
*   **Basic Training Pipeline**: A minimal training loop was successfully integrated. This included defining `nn.CrossEntropyLoss` as the loss function, `optim.Adam` as the optimizer, generating dummy target labels, and performing a single training step (zero\_grad, forward, backward, optimizer.step). The reported loss after this step was approximately $2.256.

### Insights or Next Steps

*   The core ST-GCN model architecture, as outlined in the `DeepGait_POC.pdf`, has been successfully implemented and validated with dummy data, demonstrating a robust foundation for gait recognition.
*   Further development should focus on integrating this model with actual gait data, implementing a full training and evaluation pipeline, and potentially exploring hyperparameter tuning and model optimization based on real-world performance metrics.
